# EEE3032 optional extension: pretrained deep descriptors

This notebook supports the optional **Deep Descriptors (+10)** extension. It is deliberately separate from the core starter so that PyTorch is not required for the main coursework.

Two frozen ImageNet-pretrained feature extractors are supplied:
- MobileNetV3-Small → 576-D descriptor
- SqueezeNet 1.1 → 512-D descriptor

You are **not** expected to train or fine-tune these networks. Extension credit comes from a clear experimental question, controlled evaluation and interpretation.

Google Colab is the recommended environment because PyTorch/torchvision are normally available and the official weights can be downloaded automatically on first use.


In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path

required = {"numpy":"numpy", "cv2":"opencv-python-headless", "matplotlib":"matplotlib"}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    print("Installing optional PyTorch dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from cvpr.dataset import load_split_records
from cvpr.retrieval import compute_descriptors
from cvpr.evaluation import evaluate_retrieval, per_category_summary
from cvpr.deep import (
    make_mobilenet_v3_small_extractor,
    make_squeezenet1_1_extractor,
    save_feature_cache,
    load_feature_cache,
)


## 1. Load the fixed MSRC split

Set `DATASET_SOURCE` to the extracted dataset folder or ZIP, as in the core starter notebook.


In [ ]:
DATASET_SOURCE = Path("msrc_objcategimagedatabase_v2.zip")  # change if needed
MANIFEST = PROJECT_ROOT / "split_manifest.csv"
gallery, dev, test = load_split_records(DATASET_SOURCE, manifest_path=MANIFEST, extract_dir=PROJECT_ROOT / "data")
print(len(gallery), len(dev), len(test))


## 2. Choose an encoder

The first construction of an extractor may download the official torchvision pretrained weights. Compute descriptors for the gallery and development queries, then cache them so later experiments are fast.


In [ ]:
MODEL = "mobilenet_v3_small"  # or "squeezenet1_1"
L2_NORMALIZE = False

if MODEL == "mobilenet_v3_small":
    extractor = make_mobilenet_v3_small_extractor(l2_normalize=L2_NORMALIZE)
elif MODEL == "squeezenet1_1":
    extractor = make_squeezenet1_1_extractor(l2_normalize=L2_NORMALIZE)
else:
    raise ValueError("Unknown MODEL")

gallery_deep = compute_descriptors(gallery, extractor)
dev_deep = compute_descriptors(dev, extractor)
print("Gallery descriptor matrix:", gallery_deep.shape)

CACHE_DIR = PROJECT_ROOT / "deep_cache"
CACHE_DIR.mkdir(exist_ok=True)
save_feature_cache(CACHE_DIR / f"{MODEL}_gallery.npz", gallery, gallery_deep, descriptor_name=MODEL)
save_feature_cache(CACHE_DIR / f"{MODEL}_dev.npz", dev, dev_deep, descriptor_name=MODEL)


In [ ]:
summary, rows = evaluate_retrieval(dev, dev_deep, gallery, gallery_deep, k=10)
print(summary)
for row in per_category_summary(rows, k=10):
    print(row)


## 3. Turn this into an investigation

Do not stop at the first score. Pose a question, for example:
- Does L2 normalisation change retrieval quality?
- Which encoder performs better, and for which groups?
- Does cosine-like comparison (L2-normalised vectors + Euclidean distance) change the ranking?
- Does fusing a deep score with your classical core descriptor help specific groups?

Use development queries to make choices. Reserve test queries for the final comparison, exactly as in the core coursework.
